In [ ]:
# カレントディレクトリを変更
import os
os.chdir('../')

# 必要なライブラリをインストール
# ノートブック環境では先頭に ! をつけてシェルコマンドとして実行します
!pip install snowflake-snowpark-python==1.39.0
!pip install -r requirements.txt

In [ ]:
# python再起動
%restart_python

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import glob
import importlib.util
import shutil
from snowflake.snowpark import Session

BASE_DIR = '/Workspace/Users/1210546@tmc.twfr.toyota.co.jp/congestion_and_sporty_detection'  # + 親ディレクトリ

DOTENV_ABS = f"{BASE_DIR}/.env"
TOKEN_PICKLE_ABS = f"{BASE_DIR}/token.pickle"
load_dotenv(DOTENV_ABS, override=True)    # envファイルの読み込み、同名の変数があれば上書き。
os.environ['OAUTH_TOKEN_FILE_PATH'] = TOKEN_PICKLE_ABS  # ★ Import より前に設定
MODULE_PATH = os.path.join(BASE_DIR, 'snowflake_oauth.py')
spec = importlib.util.spec_from_file_location('snowflake_oauth', MODULE_PATH)
snowflake_oauth = importlib.util.module_from_spec(spec)
spec.loader.exec_module(snowflake_oauth)

print("CWD:", os.getcwd())

os.chdir('../')
from snowflake_oauth import fetch_token, access_token

t = fetch_token()
print("token keys:", list(t.keys()))
print("access_token len:", len(access_token()))

sf_account = os.getenv("SNOWFLAKE_ACCOUNT")
sf_user = os.getenv("SNOWFLAKE_USER")
sf_database = os.getenv("SNOWFLAKE_DB")
missing = [k for k,v in {"SNOWFLAKE_ACCOUNT": sf_account, "SNOWFLAKE_USER": sf_user, "SNOWFLAKE_DB": sf_database}.items() if not v]
if missing:
    raise RuntimeError(f"必須環境変数が未設定です: {missing}")
params = {
    "account":    sf_account,
    "user":       sf_user,
    "database":   sf_database,
    "token":      access_token(),  # 文字列
    "authenticator": "oauth",
}
session = Session.builder.configs(params).create()
print("✅ Snowpark session ready")

In [ ]:
# --- handover/ を import パスへ (notebooks/ の 1つ上) ---
import sys, os, importlib
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parents[1]           # notebooks/ + handover/
sys.path.insert(0, str(PROJECT_ROOT))          # ★ src の"殻だけ"を追加 (src 自体は追加しない)

# --- .env を読み込む (上書きしない) ---
load_dotenv(PROJECT_ROOT / '.env', override=False)

# --- トークンファイルの場所 (OAUTH_TOKEN_FILE_PATH="token_pickle") ---
os.environ["OAUTH_TOKEN_FILE_PATH"] = str(PROJECT_ROOT / "token_pickle")

# --- handover 直下の snowflake_oauth を通常 import ---
_sfo = importlib.import_module("snowflake_oauth")    # handover/snowflake_oauth.py

# --- 既存コード互換: 'from src import snowflake_oauth' でも解決できるように別名登録 ---
import src as _src                                      # handover/src/__init__.py (空でOK)
setattr(_src, "snowflake_oauth", _sfo)

print("✅ bootstrap (.env only) done")

データの読み込み

In [ ]:
from src import query_utils as q

df1 = q.get_snowflake_table(
    snowflake_session=session,
    data_params={"schema_name": "VAP_SOH_ANALYTIX_1210546_TTTC_READONLY", "table_name": "VERITAS_CAN_20260306105528"}
)

df2 = q.get_snowflake_table(
    snowflake_session=session,
    data_params={"schema_name": "VAP_SOH_ANALYTIX_1210546_TTTC_READONLY", "table_name": "VERITAS_CAN_20260306110159"}
)

df3 = q.get_snowflake_table(
    snowflake_session=session,
    data_params={"schema_name": "VAP_SOH_ANALYTIX_1210546_TTTC_READONLY", "table_name": "VERITAS_CAN_20260306110603"}
)

# 例：不要列を削除（複数可）
drop_cols = ['VALUE', 'LINK_ID', 'SP1', 'ACC_BOB', 'LATITUDE', 'LONGITUDE',
             'NMLATITUDE', 'NMLONGITUDE', 'PT_DT']

crown_2411_df = df1.drop(*drop_cols)
crown_2506_df = df2.drop(*drop_cols)
crown_2601_df = df3.drop(*drop_cols)

In [ ]:
import os
import pandas as pd
import numpy as np
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, DoubleType

# ============================================
# 1. 1トリップ分の判定＆集計を行う関数
# ============================================
def process_single_trip(df: pd.DataFrame) -> pd.DataFrame:
    """
    Snowparkから1トリップ分のデータを受け取り、渋滞判定と集計を行って「1行」のデータを返す関数
    """
    # 0. データの整備とソート
    df['GPS_TIMESTAMP'] = pd.to_datetime(df['GPS_TIMESTAMP'])
    df = df.sort_values('GPS_TIMESTAMP').reset_index(drop=True)
    # 1秒間隔のリサンプリング
    df = df.set_index('GPS_TIMESTAMP').resample('1s').ffill().reset_index()

    # 1. 前処理（平滑化とフラグ作成）
    df['SPEED_smoothed'] = df['SPEED'].rolling(window=5, min_periods=1).mean()
    df['stop_confirmed'] = (df['SPEED_smoothed'] < 1.0).rolling(window=2).sum() == 2
    df['moving_confirmed'] = (df['SPEED_smoothed'] > 5.0).rolling(window=2).sum() == 2
    is_pn = df['ATSHIFTPOSITION'].isin([10, 30])
    idle_cond = (df['SPEED_smoothed'] < 1.0) & is_pn
    df['idle_120s'] = idle_cond.rolling(window=120).sum() == 120
    df['stop_edge'] = (df['stop_confirmed'].astype(int).diff() == 1).astype(int)

    # 2. 窓枠集計の計算（Window: 120s）
    df['r_low25'] = (df['SPEED_smoothed'] < 25.0).rolling(window=120).mean()
    df['stop_ep'] = df['stop_edge'].rolling(window=120).sum()
    df['has_idle_in_window'] = df['idle_120s'].rolling(window=120).max() == 1

    # 3. 評価とステートマシン（30秒ステップ）
    df['jam_eval_point'] = False
    eval_indices = range(120, len(df), 30)
    state = 'NORMAL'
    candidate_count = 0
    for i in eval_indices:
        row = df.iloc[i]
        is_candidate = (row['r_low25'] >= 0.65) and (row['stop_ep'] >= 3) and not (row['has_idle_in_window'])
        is_exit = (row['r_low25'] <= 0.45) or (row['stop_ep'] <= 1)
        if state == 'NORMAL':
            if is_candidate:
                candidate_count += 1
                if candidate_count >= 2:
                    state = 'CONGESTED'
            else:
                candidate_count = 0
        elif state == 'CONGESTED':
            if is_exit:
                state = 'NORMAL'
                candidate_count = 0

        write_idx = max(0, i - 60)
        df.loc[write_idx, 'jam_eval_point'] = (state == 'CONGESTED')

    # 4. イベント後処理
    df['is_jam_raw'] = df['jam_eval_point'].replace(False, np.nan).ffill(limit=30).fillna(False)
    # グループ化して短い断片の補正（60秒以下の断片をジャムとみなす）
    df['group_id'] = (df['is_jam_raw'] != df['is_jam_raw'].shift()).cumsum()
    group_lengths = df.groupby('group_id')['group_id'].transform('size')
    df.loc[~df['is_jam_raw'] & (group_lengths <= 60), 'is_jam_raw'] = True
    # さらに短い断片(120秒未満)をノイズとして除去
    df['group_id'] = (df['is_jam_raw'] != df['is_jam_raw'].shift()).cumsum()
    group_lengths = df.groupby('group_id')['group_id'].transform('size')
    df['is_jam_final'] = df['is_jam_raw'].copy()
    df.loc[(df['is_jam_raw']) & (group_lengths < 120), 'is_jam_final'] = False

    # ====================================================
    # 5. 出力形式の変換（1トリップにつき1行）
    # ====================================================

    # 1トリップなので1行目のVINとトリップカウントを取得
    raw_vin = df['MASKED_VIN'].iloc[0]
    vin = str(raw_vin) if pd.notna(raw_vin) else None
    raw_trip = df['TRIPCOUNT'].iloc[0]
    trip_id = int(raw_trip) if pd.notna(raw_trip) else None
    
    # トータルの走行時間と距離
    trip_total_time_sec = int(len(df))
    odometer_max = df['ODOMETER_KM'].max()
    odometer_min = df['ODOMETER_KM'].min()
    trip_total_dist_km = float(odometer_max - odometer_min) if pd.notna(odometer_max) and pd.notna(odometer_min) else 0.0

    # 渋滞エピソード（連続区間）ごとにIDを付与
    # shift() で1つ上の行と状態を比べ、「False->True」や「True->False」に状態が変化したタイミングで数字を1つ足していく（cumsum）処理
    # 連続した渋滞のかたまりごとに [1, 2, 3...] と固有のブロックID（エピソードID）が振られる。（後段で渋滞回数を数えるために使う）
    df['jam_episode_id'] = (df['is_jam_final'] != df['is_jam_final'].shift()).cumsum()
    jam_df = df[df['is_jam_final'] == True]

    # 渋滞時間と割合の計算
    trip_jam_time_sec = int(len(jam_df))
    trip_jam_ratio = float(trip_jam_time_sec / trip_total_time_sec) if trip_total_time_sec > 0 else 0.0

    result_row = {}

    if jam_df.empty:
        # 渋滞なしの場合
        result_row = {
            'MASKED_VIN': vin,
            'TRIPCOUNT': trip_id,
            'TRIP_TOTAL_TIME_SEC': trip_total_time_sec,
            'TRIP_TOTAL_DIST_KM': trip_total_dist_km,
            'TRIP_JAM_TIME_SEC': 0,
            'TRIP_JAM_DIST_KM': 0.0,
            'JAM_TIME_RATIO': 0.0,
            'TRIP_JAM_COUNT': 0,
            'JAM_EPISODES': []
        }
    else:
        # 渋滞ありの場合。渋滞中に進んだ距離
        ep_dists = jam_df.groupby('jam_episode_id')['ODOMETER_KM'].agg(lambda x: x.max() - x.min())
        trip_jam_dist_km = float(ep_dists.sum()) if not ep_dists.empty and pd.notna(ep_dists.sum()) else 0.0
    
        # uniqueなエピソードIDの数が「渋滞回数」になる
        trip_jam_count = int(jam_df['jam_episode_id'].nunique())

        result_row = {
            'MASKED_VIN': vin,
            'TRIPCOUNT': trip_id,
            'TRIP_TOTAL_TIME_SEC': trip_total_time_sec,
            'TRIP_TOTAL_DIST_KM': trip_total_dist_km,
            'TRIP_JAM_TIME_SEC': trip_jam_time_sec,
            'TRIP_JAM_DIST_KM': trip_jam_dist_km,
            'JAM_TIME_RATIO': trip_jam_ratio,
            'TRIP_JAM_COUNT': trip_jam_count
        }

    # 作成した1行だけの結果をDataFrameにして返す
    return pd.DataFrame([result_row])

In [ ]:
# ===================================================
# 2. Snowpark用のスキーマ定義（1行/トリップ版）
# ===================================================
# session は既に作成されている前提（実変名 session）
session.use_database(os.getenv("SNOWFLAKE_DB"))
session.use_schema("VAP_SCH_ANALYTIX_1640138_TTTC")

result_schema = StructType([
    StructField("MASKED_VIN", StringType()),
    StructField("TRIPCOUNT", IntegerType()),
    StructField("TRIP_TOTAL_TIME_SEC", IntegerType()),
    StructField("TRIP_TOTAL_DIST_KM", DoubleType()),
    StructField("TRIP_JAM_TIME_SEC", IntegerType()),
    StructField("TRIP_JAM_DIST_KM", DoubleType()),
    StructField("JAM_TIME_RATIO", DoubleType()),
    StructField("TRIP_JAM_COUNT", IntegerType())
])

result_spark_df = crown_2411_df.groupBy("MASKED_VIN", "TRIPCOUNT").applyInPandas(process_single_trip, output_schema=result_schema)

In [ ]:
# ===================================================
# 保存先のテーブル名を指定
# ===================================================
# ※スキーマは session.use_schema("VAP_SCH_ANALYTIX_1640138_TTTC")
table_name = "CROWNSPORT_2411_TRAFFIC_JAM_RESULT"

# ===================================================
# 2. Snowflakeのテーブルとして保存
# ===================================================
# mode("overwrite"): 既に同じ名前のテーブルがあれば（上書き（作り直し））
# mode("append"): 既にテーブルがあり、そこに新しいデータを「追記」したい場合

result_spark_df.write.mode("overwrite").save_as_table(table_name)
print(f"Snowflakeテーブル `{table_name}` に集計結果の保存が完了しました。")

In [ ]:
from src import query_utils as q
result_crom2411 = q.get_snowflake_table(
    snowflake_session=session,
    data_params={"schema_name": 'VAP_SOH_ANALYTIX_1640138_TTTC', "table_name": 'CROMSPORT_2411_TRAFFIC_JAM_RESULT'}
)
df_summary = result_crom2411.toPandas()
print(df_summary.shape)
df_summary.head()